In [1]:
!pip install --upgrade pyproject-hooks build


[notice] A new release of pip available: 22.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install --use-pep517 kaggle

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for kaggle: filename=kaggle-1.6.17-py3-none-any.whl size=105840 sha256=f86b85d4ac14600bbb1415ab20161803c249d13206ad35127e238c6c501ce4e2
  Stored in directory: /tmp/pip-ephem-wheel-cache-1qssufhp/wheels/2b/af/a9/70bffa2773af622d2ebea9c8d407720b86e67bd40c465bf837
Successfully built kaggle

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import os

# Crea la carpeta .kaggle en el home si no existe
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

# Mueve el kaggle.json subido a ~/.kaggle/kaggle.json
!mv kaggle.json ~/.kaggle/kaggle.json

# Ajusta permisos a 600
!chmod 600 ~/.kaggle/kaggle.json

print("kaggle.json configurado en ~/.kaggle/")


kaggle.json configurado en ~/.kaggle/


In [6]:
!rm -rf /tmp/data/*
!echo "✅ Archivos antiguos eliminados."


✅ Archivos antiguos eliminados.


In [7]:
# 1️⃣ Crear la carpeta de trabajo
!mkdir -p /tmp/data

# 2️⃣ Descargar el dataset completo
!kaggle datasets download -d marcosdiezz/consumo-electrico-0 -p /tmp/data

# 3️⃣ Descomprimir solo el archivo deseado
!unzip -o "/tmp/data/consumo-electrico-0.zip" "divididof/consumo_parte_116.csv" -d /tmp/data

# 4️⃣ Mover el archivo al directorio raíz
!mv /tmp/data/divididof/consumo_parte_116.csv /tmp/data/

# 5️⃣ Eliminar los archivos no deseados para limpiar espacio
!rm -rf /tmp/data/divididof /tmp/data/consumo-electrico-0.zip

# 6️⃣ Verificar que solo queda el archivo correcto
!ls -lh /tmp/data

Dataset URL: https://www.kaggle.com/datasets/marcosdiezz/consumo-electrico-0
License(s): unknown
100%|██████████████████████████████████████| 11.1G/11.1G [13:49<00:00, 21.8MB/s]
100%|██████████████████████████████████████| 11.1G/11.1G [13:49<00:00, 14.3MB/s]
Archive:  /tmp/data/consumo-electrico-0.zip
  inflating: /tmp/data/divididof/consumo_parte_116.csv  
total 35M
-rw-r--r-- 1 jovyan users 35M Mar  4 16:11 consumo_parte_116.csv


In [8]:
import mlrun
import pandas as pd

# Ruta del dataset seleccionado
csv_path = "/tmp/data/consumo_parte_116.csv"

# Cargar el CSV en un DataFrame
df = pd.read_csv(csv_path)

# Crear un contexto de MLRun
context = mlrun.get_or_create_ctx("kaggle-ingest")

# Loggear el dataset en MLRun para que se guarde en MinIO
context.log_dataset("consumo_electrico", df=df, format="csv")

print("✅ Dataset `consumo_parte_116.csv` subido a MLRun y MinIO.")

> 2025-03-05 18:09:25,674 [info] logging run results to: http://mlrun-api:8080
✅ Dataset `consumo_parte_116.csv` subido a MLRun y MinIO.


In [1]:
%%writefile preprocess.py
import mlrun
from storey import MapClass
import pandas as pd

def preprocess_data(context, file_path: str):
    """Preprocesa y registra datos en el Feature Store."""
    
    # Cargar el dataset
    df = pd.read_csv(file_path)

    # Convertir fecha y hora a formato datetime
    df["fecha_hora"] = pd.to_datetime(df["fecha"] + " " + df["hora"])

    # Eliminar columnas originales de fecha y hora
    df.drop(columns=["fecha", "hora"], inplace=True)

    # Eliminar filas con valores nulos
    df.dropna(inplace=True)

    # Filtrar valores negativos en consumo_kwh y coste_euros
    df = df[(df["consumo_kwh"] > 0) & (df["coste_euros"] > 0)]

    # Obtener el proyecto de MLRun
    project = mlrun.get_or_create_project("smartgrids")

    # Definir el Feature Store
    feature_set = project.get_feature_set(
        name="consumo_electrico",
        entities=["casa_id", "fecha_hora"],  # Llaves primarias
        description="Consumo eléctrico por casa y tiempo"
    )

    # Ingerir los datos en el Feature Store
    feature_set.ingest(df)

    # Guardar dataset preprocesado en Parquet
    context.log_dataset("consumo_electrico", df=df, format="parquet")

    print("✅ Datos preprocesados y almacenados en el Feature Store.")

Overwriting preprocess.py


In [2]:
import mlrun
from mlrun import code_to_function

# Crear el proyecto en MLRun
project = mlrun.get_or_create_project("smartgrids", context="./")

# Registrar la función de preprocesamiento
preprocess_func = code_to_function(
    name="preprocess-data",
    filename="preprocess.py",
    handler="preprocess_data",
    kind="job",
    image="mlrun/mlrun"
)

project.set_function(preprocess_func)
project.save()

> 2025-03-10 10:19:51,691 [info] Project loaded successfully: {"project_name":"smartgrids"}


In [6]:
# Recuperar la última versión del dataset desde MLRun
data_item = project.get_artifact("consumo_electrico")

# Obtener la ruta real del archivo almacenado en MinIO
file_path = data_item.get_target_path()

run = preprocess_func.run(
    inputs={"file_path": file_path},  # Se usa la ruta de MinIO obtenida desde MLRun
    local=True
)

MLRunNotFoundError: read artifact smartgrids/consumo_electrico details: MLRunNotFoundError('Artifact smartgrids/consumo_electrico:latest not found')

In [ ]:
run = preprocess_func.run(
    inputs={"file_path": mlrun.get_dataitem("store://datasets/consumo_electrico").get_target_path()},
    local=True
)